# Train your first 🐸 TTS model 💫

### 👋 Hello and welcome to Coqui (🐸) TTS

The goal of this notebook is to show you a **typical workflow** for **training** and **testing** a TTS model with 🐸.

Let's train a very small model on a very small amount of data so we can iterate quickly.

In this notebook, we will:

1. Download data and format it for 🐸 TTS.
2. Configure the training and testing runs.
3. Train a new model.
4. Test the model and display its performance.

So, let's jump right in!


In [4]:
## Install Coqui TTS
! pip install -U pip
! pip install TTS

## ✅ Data Preparation

### **First things first**: we need some data.

We're training a Text-to-Speech model, so we need some _text_ and we need some _speech_. Specificially, we want _transcribed speech_. The speech must be divided into audio clips and each clip needs transcription. More details about data requirements such as recording characteristics, background noise and vocabulary coverage can be found in the [🐸TTS documentation](https://tts.readthedocs.io/en/latest/formatting_your_dataset.html).

If you have a single audio file and you need to **split** it into clips. It is also important to use a lossless audio file format to prevent compression artifacts. We recommend using **wav** file format.

The data format we will be adopting for this tutorial is taken from the widely-used  **LJSpeech** dataset, where **waves** are collected under a folder:

<span style="color:purple;font-size:15px">
/wavs<br />
 &emsp;| - audio1.wav<br />
 &emsp;| - audio2.wav<br />
 &emsp;| - audio3.wav<br />
  ...<br />
</span>

and a **metadata.csv** file will have the audio file name in parallel to the transcript, delimited by `|`:

<span style="color:purple;font-size:15px">
# metadata.csv <br />
audio1|This is my sentence. <br />
audio2|This is maybe my sentence. <br />
audio3|This is certainly my sentence. <br />
audio4|Let this be your sentence. <br />
...
</span>

In the end, we should have the following **folder structure**:

<span style="color:purple;font-size:15px">
/MyTTSDataset <br />
&emsp;| <br />
&emsp;| -> metadata.csv<br />
&emsp;| -> /wavs<br />
&emsp;&emsp;| -> audio1.wav<br />
&emsp;&emsp;| -> audio2.wav<br />
&emsp;&emsp;| ...<br />
</span>

🐸TTS already provides tooling for the _LJSpeech_. if you use the same format, you can start training your models right away. <br />

After you collect and format your dataset, you need to check two things. Whether you need a **_formatter_** and a **_text_cleaner_**. <br /> The **_formatter_** loads the text file (created above) as a list and the **_text_cleaner_** performs a sequence of text normalization operations that converts the raw text into the spoken representation (e.g. converting numbers to text, acronyms, and symbols to the spoken format).

If you use a different dataset format then the LJSpeech or the other public datasets that 🐸TTS supports, then you need to write your own **_formatter_** and  **_text_cleaner_**.

## ⏳️ Loading your dataset
Load one of the dataset supported by 🐸TTS.

We will start by defining dataset config and setting LJSpeech as our target dataset and define its path.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os

# BaseDatasetConfig: defines name, formatter and path of the dataset.
from TTS.tts.configs.shared_configs import BaseDatasetConfig

output_path = "drive/MyDrive/DatasetIKB/TeleDataNew/"
# if not os.path.exists(output_path):
#     os.removedirs()


In [ ]:
# Download and extract LJSpeech dataset.

# !wget -O $output_path/LJSpeech-1.1.tar.bz2 https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
# !tar -xf $output_path/LJSpeech-1.1.tar.bz2 -C $output_path

In [ ]:
def my_custom_formatter(root_path, meta_file):
    """
    Custom formatter untuk dataset dengan format:
    utt_id|text_asli|text_normalisasi
    """
    items = []
    meta_path = os.path.join(root_path, meta_file)

    with open(meta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            utt_id, raw_text, norm_text = line.split("|")

            audio_file = os.path.join(root_path, "wavs", f"{utt_id}.wav")
            if not os.path.exists(audio_file):
                raise FileNotFoundError(f"File audio tidak ditemukan: {audio_file}")

            items.append({
                "audio_file": audio_file,
                "text": norm_text,
                "raw_text": raw_text,
                "speaker_name": "default"
            })

    return items

In [7]:
# dataset_config = BaseDatasetConfig(
#     formatter="ljspeech", meta_file_train="metadata.txt", path=output_path
# )

dataset_config = BaseDatasetConfig(
    formatter="ljspeech",  # Pass the custom formatter function itself
    meta_file_train="metadata.txt",  # Replace with the actual name of your training metadata file
    # meta_file_val="metadata_val.txt", # Uncomment and replace if you have a validation metadata file
    path=output_path  # Replace with the actual path to your dataset directory
)

## ✅ Train a new model

Let's kick off a training run 🚀🚀🚀.

Deciding on the model architecture you'd want to use is based on your needs and available resources. Each model architecture has it's pros and cons that define the run-time efficiency and the voice quality.
We have many recipes under `TTS/recipes/` that provide a good starting point. For this tutorial, we will be using `GlowTTS`.

We will begin by initializing the model training configuration.

In [8]:
# GlowTTSConfig: all model related values for training, validating and testing.
from TTS.tts.configs.glow_tts_config import GlowTTSConfig
config = GlowTTSConfig(
    batch_size=32,
    eval_batch_size=16,
    num_loader_workers=4,
    num_eval_loader_workers=4,
    run_eval=True,
    test_delay_epochs=-1,
    epochs=10,
    text_cleaner="phoneme_cleaners",
    phoneme_language="id",
    phoneme_cache_path=os.path.join(output_path, "phoneme_cache"),
    print_step=25,
    print_eval=False,
    mixed_precision=True,
    output_path=output_path,
    datasets=[dataset_config],
    save_step=1000,
)

Next we will initialize the audio processor which is used for feature extraction and audio I/O.

In [9]:
from TTS.utils.audio import AudioProcessor
ap = AudioProcessor.init_from_config(config)
# Modify sample rate if for a custom audio dataset:
ap.sample_rate = 22050


 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:True
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:45
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024


Next we will initialize the tokenizer which is used to convert text to sequences of token IDs.  If characters are not defined in the config, default characters are passed to the config.

In [10]:
from TTS.tts.utils.text.tokenizer import TTSTokenizer
tokenizer, config = TTSTokenizer.init_from_config(config)

Next we will load data samples. Each sample is a list of ```[text, audio_file_path, speaker_name]```. You can define your custom sample loader returning the list of samples.

In [11]:
from TTS.tts.datasets import load_tts_samples
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)

 | > Found 4000 files in /content/drive/MyDrive/DatasetIKB/TeleDataNew


Now we're ready to initialize the model.

Models take a config object and a speaker manager as input. Config defines the details of the model like the number of layers, the size of the embedding, etc. Speaker manager is used by multi-speaker models.

In [12]:
from TTS.tts.models.glow_tts import GlowTTS
model = GlowTTS(config, ap, tokenizer, speaker_manager=None)

Trainer provides a generic API to train all the 🐸TTS models with all its perks like mixed-precision training, distributed training, etc.

In [13]:
from trainer import Trainer, TrainerArgs
trainer = Trainer(
    TrainerArgs(), config, output_path, model=model, train_samples=train_samples, eval_samples=eval_samples
)

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: fp16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 12
 | > Num. of Torch Threads: 6
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=drive/MyDrive/DatasetIKB/TeleDataNew/run-August-13-2025_02+56AM-0000000
/usr/local/lib/python3.11/dist-packages/trainer/trainer.py:552: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()

 > Model has 28597969 parameters


### AND... 3,2,1... START TRAINING 🚀🚀🚀

In [14]:
trainer.fit()


 > EPOCH: 0/10
 --> drive/MyDrive/DatasetIKB/TeleDataNew/run-August-13-2025_02+56AM-0000000




> DataLoader initialization
| > Tokenizer:
	| > add_blank: False
	| > use_eos_bos: False
	| > use_phonemes: False
| > Number of instances : 3960



 > TRAINING (2025-08-13 02:57:01) 


 | > Preprocessing samples
 | > Max text length: 172
 | > Min text length: 35
 | > Avg text length: 99.33257575757575
 | 
 | > Max audio length: 489902.0
 | > Min audio length: 71342.0
 | > Avg audio length: 184244.94343434344
 | > Num. instances discarded samples: 0
 | > Batch group size: 0.



   --> TIME: 2025-08-13 02:57:23 -- STEP: 0/124 -- GLOBAL_STEP: 0
     | > current_lr: 2.5e-07 
     | > step_time: 2.2093  (2.209319829940796)
     | > loader_time: 19.652  (19.652039527893066)

 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
/usr/local/lib/python3.11/dist-packages/TTS/tts/models/glow_tts.py:415: FutureWarning: `torch.cuda.amp.autocast(args...)` i

...


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.0026439428329467773 (+0.00032913684844970703)
     | > avg_loss: 3.0348329544067383 (-0.17070364952087402)
     | > avg_log_mle: 0.608189195394516 (-0.04485815763473511)
     | > avg_loss_dur: 2.4266437292099 (-0.1258455514907837)

 > BEST MODEL : drive/MyDrive/DatasetIKB/TeleDataNew/run-August-13-2025_02+56AM-0000000/best_model_992.pth

 > EPOCH: 8/10
 --> drive/MyDrive/DatasetIKB/TeleDataNew/run-August-13-2025_02+56AM-0000000

 > TRAINING (2025-08-13 03:15:29) 

   --> TIME: 2025-08-13 03:15:38 -- STEP: 8/124 -- GLOBAL_STEP: 1000
     | > loss: 3.493443012237549  (3.457059532403946)
     | > log_mle: 0.6040027141571045  (0.611593134701252)
     | > loss_dur: 2.8894402980804443  (2.845466375350952)
     | > amp_scaler: 16384.0  (16384.0)
     | > grad_norm: tensor(6.2218, device='cuda:0')  (tensor(6.1854, device='cuda:0'))
     | > current_lr: 2e-06 
     | > step_time: 0.4751  (0.5131531059741974)
     | > loader_time: 0.005  (0.007

 | > Synthesizing test sentences.



  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.0027265548706054688 (+8.26120376586914e-05)
     | > avg_loss: 2.7984498739242554 (-0.2363830804824829)
     | > avg_log_mle: 0.5594853460788727 (-0.04870384931564331)
     | > avg_loss_dur: 2.2389644384384155 (-0.18767929077148438)

 > BEST MODEL : drive/MyDrive/DatasetIKB/TeleDataNew/run-August-13-2025_02+56AM-0000000/best_model_1116.pth

 > EPOCH: 9/10
 --> drive/MyDrive/DatasetIKB/TeleDataNew/run-August-13-2025_02+56AM-0000000

 > TRAINING (2025-08-13 03:17:02) 

   --> TIME: 2025-08-13 03:17:11 -- STEP: 9/124 -- GLOBAL_STEP: 1125
     | > loss: 3.0534205436706543  (3.220828056335449)
     | > log_mle: 0.5554563999176025  (0.5633339285850525)
     | > loss_dur: 2.4979641437530518  (2.6574940946367054)
     | > amp_scaler: 16384.0  (16384.0)
     | > grad_norm: tensor(5.4317, device='cuda:0')  (tensor(5.6851, device='cuda:0'))
     | > current_lr: 2.25e-06 
     | > step_time: 0.4699  (0.4938256475660536)
     | > loader_time: 0.00

 | > Synthesizing test sentences.



  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.0024101734161376953 (-0.00031638145446777344)
     | > avg_loss: 2.5705201625823975 (-0.2279297113418579)
     | > avg_log_mle: 0.5071526169776917 (-0.05233272910118103)
     | > avg_loss_dur: 2.063367486000061 (-0.1755969524383545)

 > BEST MODEL : drive/MyDrive/DatasetIKB/TeleDataNew/run-August-13-2025_02+56AM-0000000/best_model_1240.pth


#### 🚀 Run the Tensorboard. 🚀
On the notebook and Tensorboard, you can monitor the progress of your model. Also Tensorboard provides certain figures and sample outputs.

In [16]:
!pip install tensorboard
!tensorboard --logdir=tts_train_dir

2025-08-13 03:20:10.315399: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755055210.337061   18234 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755055210.343644   18234 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755055210.360193   18234 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755055210.360219   18234 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755055210.360222   18234 computation_placer.cc:177] computation placer alr

## ✅ Test the model

We made it! 🙌

Let's kick off the testing run, which displays performance metrics.

We're committing the cardinal sin of ML 😈 (aka - testing on our training data) so you don't want to deploy this model into production. In this notebook we're focusing on the workflow itself, so it's forgivable 😇

You can see from the test output that our tiny model has overfit to the data, and basically memorized this one sentence.

When you start training your own models, make sure your testing data doesn't include your training data 😅

Let's get the latest saved checkpoint.

In [17]:
import glob, os
# output_path = "tts_train_dir"
ckpts = sorted([f for f in glob.glob(output_path+"/*/*.pth")])
configs = sorted([f for f in glob.glob(output_path+"/*/*.json")])

In [22]:
!tts --text "Halo semuanya, apa kabar? Kenalin nih aku raja, kamu jelata" \
      --model_path {ckpts[-1]} \
      --config_path {configs[-1]} \
      --out_path out.wav

 > Using model: glow_tts
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:True
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:45
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Text: Halo semuanya, apa kabar? Kenalin nih aku raja, kamu jelata
 > Text splitted to sentences.
['Halo semuanya, apa kabar?', 'Kenalin nih aku raja, kamu jelata']
 > Processing time: 0.6248843669891357
 > Real-time factor: 0.4073

## 📣 Listen to the synthesized wave 📣

In [23]:
import IPython
IPython.display.Audio("out.wav")

## 🎉 Congratulations! 🎉 You now have trained your first TTS model!
Follow up with the next tutorials to learn more advanced material.